In [ ]:
import sys
import os
sys.path.append(os.path.abspath(".."))

import pandas as pd
import matplotlib.pyplot as plt

from src.data.market_loader import MarketLoader
from src.data.synthetic_sofr_builder import build_term_sofr_curve

from src.term_structure.bootstrap_curve import (
    bootstrap_df_from_sofr,
    bootstrap_df_from_treasury_curve
)
from src.term_structure.curve_interpolation import (
    build_coupon_structure,
    zero_rate_to_df,
    interpolate_zero_curve,
    convert_tenor_columns_to_yearfrac,
    log_linear_interpolation_dfs
)

from src.term_structure.zero_curve_builder import (
    build_zero_curve_from_sofr_dfs,
    build_zero_curve_from_discount_curve
)

from src.term_structure.forward_curve_builder import build_forward_curve_from_discount_curve

In [2]:
# downloading market curves
loader = MarketLoader()
curves = loader.loader_pipeline()

# building synthetic sofr curve from ON rates and futures
sofr_curve = build_term_sofr_curve(curves = curves)
sofr_curve

treasury curve dataset already downloaded..
sofr curve dataset already downloaded..
futures curve dataset already downloaded..


,ON,3M,6M,1M
Date,,,,
2018-04-03,1.83,1.72,1.88,1.79
2018-04-04,1.74,1.68,1.86,1.72
2018-04-05,1.75,1.69,1.88,1.73
2018-04-06,1.75,1.70,1.86,1.73
2018-04-09,1.75,1.73,1.89,1.74
...,...,...,...,...
2026-04-15,3.72,3.62,3.60,3.69
2026-04-16,3.67,3.62,3.58,3.65
2026-04-17,3.65,3.61,3.56,3.64


In [ ]:
# bootstrapping discount factors from sofr curve
df_bootstrap_from_sofr = bootstrap_df_from_sofr(
    sofr_curve = sofr_curve
)

# constructing semi-annual coupon structure
coupon_structure = build_coupon_structure(
    max_year = 30,
    freq = 2
)

# numerically sorted df curve with yearfrac columns
df_bootstrap_from_sofr_yearfrac = convert_tenor_columns_to_yearfrac(
    df_curve = df_bootstrap_from_sofr
)

# log-linear interpolation of discount curve
df_mm_interp = log_linear_interpolation_dfs(
    df_curve = df_bootstrap_from_sofr_yearfrac,
    target_times = coupon_structure
)

ValueError: could not convert string to float: 'ON'

In [ ]:
# constructing zero curve from discount curve
zero_curve_mm = build_zero_curve_from_sofr_dfs(
    discount_curve = df_bootstrap_from_sofr
)

zero_curve_mm

In [ ]:
# interpolate zero curve to coupon structure
coupon_structure = build_coupon_structure(
    max_year = 30,
    freq = 2
)

zero_curve_mm_interp = interpolate_zero_curve(
    zero_curve_df = zero_curve_mm,
    target_times = coupon_structure
)

### short-end of the curve
# df bootstrap using interpolated zero curve
df_mm_interp = zero_rate_to_df(
    zero_curve_interp = zero_curve_mm_interp
)

### long-end of the curve -> constructing discount factors from treasury par yields
# get treasury par yields
treasury_curve = curves['treasury']

# bootstrapping discount factors from treasury par yields merging money market dfs
df_bootstrap_full = bootstrap_df_from_treasury_curve(
    treasury_curve = treasury_curve,
    interpolated_dfs = df_mm_interp
)

# building zero curve from bootstrapped discount curve
zero_curve_full = build_zero_curve_from_discount_curve(
    discount_curve = df_bootstrap_full
)


# building forward curve from bootstrapped discount curve
forward_curve_full = build_forward_curve_from_discount_curve(
    discount_curve = df_bootstrap_full
) 


In [ ]:
# plotting curves for a sample date
sample_date = df_bootstrap_full.index[-1]

fig, axes = plt.subplots(3, 1, figsize=(12, 10))

axes[0].plot(df_bootstrap_full.columns, df_bootstrap_full.loc[sample_date])
axes[0].set_title("Discount Curve", fontweight = "bold")
axes[0].set_ylabel("DF")

axes[1].plot(zero_curve_full.columns, zero_curve_full.loc[sample_date])
axes[1].set_title("Zero Curve", fontweight = "bold")
axes[1].set_ylabel("Rate")

axes[2].plot(forward_curve_full.columns, forward_curve_full.loc[sample_date])
axes[2].set_title("Forward Curve", fontweight = "bold")
axes[2].set_ylabel("Rate")

plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
a = np.array([1, 2, 5, 7, 10])
np.searchsorted(a, 6)